# 14 — Role-Playing Dimension + Degenerate Dimension — DuckDB

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())
from utils import get_conn, DB_PATH, DATA_DIR

conn = get_conn()
print(f"Conectado: {DB_PATH}")

Conectado: /workspace/pf_northwind/duckdb/northwind_dw.duckdb


In [2]:
# ============================================================
# Role-Playing Views sobre DimDate
# ============================================================
conn.execute("CREATE OR REPLACE VIEW gold.v_order_date    AS SELECT * FROM gold.DimDate")
conn.execute("CREATE OR REPLACE VIEW gold.v_required_date AS SELECT * FROM gold.DimDate")
conn.execute("CREATE OR REPLACE VIEW gold.v_shipped_date  AS SELECT * FROM gold.DimDate")

for view in ["gold.v_order_date", "gold.v_required_date", "gold.v_shipped_date"]:
    n = conn.execute(f"SELECT COUNT(*) AS n FROM {view}").fetchdf()['n'][0]
    base = conn.execute("SELECT COUNT(*) AS n FROM gold.DimDate").fetchdf()['n'][0]
    assert n == base, f"{view} diverge de DimDate"
print("✓ Role-playing views OK")


✓ Role-playing views OK


In [3]:
# ============================================================
# DEMO Role-Playing: 3 aliases do mesmo DimDate
# ============================================================
conn.execute("""
    SELECT od.Year, od.Quarter,
           COUNT(DISTINCT f.OrderID) AS TotalPedidos,
           ROUND(AVG(f.DaysToShip::DOUBLE), 1) AS MediaDias,
           SUM(CASE WHEN f.IsLate THEN 1 ELSE 0 END) AS Atrasados
    FROM gold.FactOrderFulfillment f
    JOIN gold.v_order_date    od ON od.DateKey = f.OrderDateKey
    LEFT JOIN gold.v_shipped_date  sd ON sd.DateKey = f.ShippedDateKey
    LEFT JOIN gold.v_required_date rd ON rd.DateKey = f.RequiredDateKey
    GROUP BY od.Year, od.Quarter
    ORDER BY od.Year, od.Quarter
""").fetchdf()


,Year,Quarter,TotalPedidos,MediaDias,Atrasados
0,1996,3,70,8.9,5
1,1996,4,82,7.5,2
2,1997,1,92,9.2,5
3,1997,2,93,9.0,4
4,1997,3,103,8.2,5
5,1997,4,120,9.2,8
6,1998,1,182,8.6,8
7,1998,2,88,6.4,0


In [4]:
# ============================================================
# DEMO Degenerate Dimension: drill-through via OrderID
# ============================================================
conn.execute("""
    SELECT fs.OrderID,  -- Degenerate Dimension
           dc.CompanyName,
           dp.ProductName,
           fs.Quantity,
           fs.NetRevenue
    FROM gold.FactSales fs
    JOIN gold.DimCustomer dc ON dc.CustomerSK = fs.CustomerSK AND dc.IsCurrent
    JOIN gold.DimProduct  dp ON dp.ProductSK  = fs.ProductSK  AND dp.IsCurrent
    WHERE fs.OrderID = 10248
""").fetchdf()


,OrderID,CompanyName,ProductName,Quantity,NetRevenue
0,10248,Vins et alcools Chevalier,Mozzarella di Giovanni,5,174.0
1,10248,Vins et alcools Chevalier,Singaporean Hokkien Fried Mee,10,98.0
2,10248,Vins et alcools Chevalier,Queso Cabrales,12,168.0
